In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

In [2]:
# Prevent PIL from crashing on truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [3]:
# CHANGE THIS to your actual Kaggle dataset path
DATA_ROOT = Path("/kaggle/input/datasets/zuhaaqib/wang-cnndetection-dataset/val/val")

# Allowed extensions
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

print("Dataset root exists:", DATA_ROOT.exists())
print("Dataset root:", DATA_ROOT)

Dataset root exists: True
Dataset root: /kaggle/input/datasets/zuhaaqib/wang-cnndetection-dataset/val/val


In [4]:
def collect_labeled_images(root_dir: Path):
    """
    Recursively collect images.
    If image is under folder named 0_real -> label 0
    If image is under folder named 1_fake -> label 1
    """
    samples = []

    for file_path in root_dir.rglob("*"):
        if not file_path.is_file():
            continue
        if file_path.suffix.lower() not in VALID_EXTENSIONS:
            continue

        parent_names = {p.name for p in file_path.parents}

        if "0_real" in parent_names:
            samples.append((str(file_path), 0))
        elif "1_fake" in parent_names:
            samples.append((str(file_path), 1))

    return samples


samples = collect_labeled_images(DATA_ROOT)

print("Total images found:", len(samples))
print("First 5 samples:", samples[:5])

num_real = sum(label == 0 for _, label in samples)
num_fake = sum(label == 1 for _, label in samples)

print("Real images:", num_real)
print("Fake images:", num_fake)

assert len(samples) > 0, "No images found. Check DATA_ROOT."
assert num_real > 0 and num_fake > 0, "Both 0_real and 1_fake must be present."

Total images found: 8000
First 5 samples: [('/kaggle/input/datasets/zuhaaqib/wang-cnndetection-dataset/val/val/motorbike/1_fake/03092.png', 1), ('/kaggle/input/datasets/zuhaaqib/wang-cnndetection-dataset/val/val/motorbike/1_fake/03508.png', 1), ('/kaggle/input/datasets/zuhaaqib/wang-cnndetection-dataset/val/val/motorbike/1_fake/05261.png', 1), ('/kaggle/input/datasets/zuhaaqib/wang-cnndetection-dataset/val/val/motorbike/1_fake/17325.png', 1), ('/kaggle/input/datasets/zuhaaqib/wang-cnndetection-dataset/val/val/motorbike/1_fake/10874.png', 1)]
Real images: 4000
Fake images: 4000


In [5]:
import os
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets/ambarish/wangdataset")  # your current path

print("Exists:", DATA_ROOT.exists())
print("Listing first few folders/files:\n")

count = 0
for root, dirs, files in os.walk(DATA_ROOT):
    print("ROOT:", root)
    print("DIRS:", dirs[:10])
    print("FILES:", files[:10])
    print("-" * 60)
    count += 1
    if count == 15:   # stop after a few levels
        break

Exists: False
Listing first few folders/files:



In [6]:
paths = [p for p, y in samples]
labels = [y for p, y in samples]

train_paths, val_paths, train_labels, val_labels = train_test_split(
    paths,
    labels,
    test_size=0.2,
    random_state=SEED,
    stratify=labels
)

train_samples = list(zip(train_paths, train_labels))
val_samples = list(zip(val_paths, val_labels))

print("Train samples:", len(train_samples))
print("Validation samples:", len(val_samples))

Train samples: 6400
Validation samples: 1600


In [7]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [8]:
class WangSubsetDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32), img_path

In [9]:
BATCH_SIZE = 32
NUM_WORKERS = 2

train_dataset = WangSubsetDataset(train_samples, transform=train_transform)
val_dataset = WangSubsetDataset(val_samples, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Train batches: 200
Val batches: 50


In [10]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# Replace final layer for binary classification
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 1)

model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print(model.classifier)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 143MB/s]


Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1, bias=True)
)


In [11]:
def evaluate_model(model, loader, device):
    model.eval()

    all_labels = []
    all_probs = []
    all_preds = []

    running_loss = 0.0

    with torch.no_grad():
        for images, labels, _ in tqdm(loader, desc="Evaluating", leave=False):
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images).squeeze(1)
            loss = criterion(logits, labels)

            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).long()

            running_loss += loss.item()

            all_labels.extend(labels.cpu().numpy().astype(int).tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())

    avg_loss = running_loss / max(len(loader), 1)

    metrics = {
        "loss": avg_loss,
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, zero_division=0),
        "recall": recall_score(all_labels, all_preds, zero_division=0),
        "f1": f1_score(all_labels, all_preds, zero_division=0),
        "roc_auc": roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0,
        "confusion_matrix": confusion_matrix(all_labels, all_preds)
    }

    return metrics

In [12]:
NUM_EPOCHS = 5
best_f1 = -1.0
history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for images, labels, _ in loop:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        logits = model(images).squeeze(1)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        loop.set_postfix(train_loss=loss.item())

    train_loss = running_loss / max(len(train_loader), 1)
    val_metrics = evaluate_model(model, val_loader, DEVICE)

    row = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
        "val_roc_auc": val_metrics["roc_auc"]
    }
    history.append(row)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss   : {train_loss:.4f}")
    print(f"Val Loss     : {val_metrics['loss']:.4f}")
    print(f"Val Accuracy : {val_metrics['accuracy']:.4f}")
    print(f"Val Precision: {val_metrics['precision']:.4f}")
    print(f"Val Recall   : {val_metrics['recall']:.4f}")
    print(f"Val F1       : {val_metrics['f1']:.4f}")
    print(f"Val ROC-AUC  : {val_metrics['roc_auc']:.4f}")
    print("Confusion Matrix:")
    print(val_metrics["confusion_matrix"])

    if val_metrics["f1"] > best_f1:
        best_f1 = val_metrics["f1"]
        torch.save(model.state_dict(), "/kaggle/working/best_efficientnet_b0_wang1000.pth")
        print("Best model saved.\n")

Epoch 1/5:   0%|          | 0/200 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]


Epoch 1
Train Loss   : 0.4190
Val Loss     : 0.1356
Val Accuracy : 0.9587
Val Precision: 0.9476
Val Recall   : 0.9712
Val F1       : 0.9593
Val ROC-AUC  : 0.9940
Confusion Matrix:
[[757  43]
 [ 23 777]]
Best model saved.



Epoch 2/5:   0%|          | 0/200 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

...

Exception ignored in: self._shutdown_workers()<function _MultiProcessingDataLoaderIter.__del__ at 0x7811c597eca0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Exception ignored in:     <function _MultiProcessingDataLoaderIter.__del__ at 0x7811c597eca0>

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
        self._shutdown_workers()if w.is_alive():

    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
      if w.is_alive(): 
    ^  ^ ^  ^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'

  File "/usr/lib/python


Epoch 4
Train Loss   : 0.0566
Val Loss     : 0.0322
Val Accuracy : 0.9900
Val Precision: 0.9937
Val Recall   : 0.9862
Val F1       : 0.9900
Val ROC-AUC  : 0.9996
Confusion Matrix:
[[795   5]
 [ 11 789]]
Best model saved.



Epoch 5/5:   0%|          | 0/200 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]


Epoch 5
Train Loss   : 0.0468
Val Loss     : 0.0308
Val Accuracy : 0.9906
Val Precision: 0.9950
Val Recall   : 0.9862
Val F1       : 0.9906
Val ROC-AUC  : 0.9995
Confusion Matrix:
[[796   4]
 [ 11 789]]
Best model saved.



In [13]:
history_df = pd.DataFrame(history)
history_df

,epoch,train_loss,val_loss,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc
0,1,0.419045,0.135574,0.958750,0.947561,0.97125,0.959259,0.993977
1,2,0.132155,0.062060,0.981250,0.976485,0.98625,0.981343,0.998820
2,3,0.084664,0.049017,0.980000,0.987310,0.97250,0.979849,0.999106
3,4,0.056554,0.032228,0.990000,0.993703,0.98625,0.989962,0.999592
4,5,0.046755,0.030836,0.990625,0.994956,0.98625,0.990584,0.999523


In [14]:
best_model = models.efficientnet_b0(weights=None)
in_features = best_model.classifier[1].in_features
best_model.classifier[1] = nn.Linear(in_features, 1)

best_model.load_state_dict(torch.load("/kaggle/working/best_efficientnet_b0_wang1000.pth", map_location=DEVICE))
best_model = best_model.to(DEVICE)
best_model.eval()

rows = []

with torch.no_grad():
    for images, labels, paths in tqdm(val_loader, desc="Predicting on val"):
        images = images.to(DEVICE)
        logits = best_model(images).squeeze(1)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long()

        for path, label, prob, pred in zip(paths, labels.numpy(), probs.cpu().numpy(), preds.cpu().numpy()):
            rows.append({
                "image_path": path,
                "true_label": int(label),
                "pred_prob_fake": float(prob),
                "pred_label": int(pred)
            })

pred_df = pd.DataFrame(rows)
pred_df.head()

Predicting on val:   0%|          | 0/50 [00:00<?, ?it/s]

,image_path,true_label,pred_prob_fake,pred_label
0,/kaggle/input/datasets/zuhaaqib/wang-cnndetect...,1,0.999981,1
1,/kaggle/input/datasets/zuhaaqib/wang-cnndetect...,0,0.295028,0
2,/kaggle/input/datasets/zuhaaqib/wang-cnndetect...,1,0.999418,1
3,/kaggle/input/datasets/zuhaaqib/wang-cnndetect...,1,0.999106,1
4,/kaggle/input/datasets/zuhaaqib/wang-cnndetect...,0,0.000407,0


In [15]:
history_df.to_csv("/kaggle/working/training_history.csv", index=False)
pred_df.to_csv("/kaggle/working/validation_predictions.csv", index=False)

print("Saved files:")
print("/kaggle/working/best_efficientnet_b0_wang1000.pth")
print("/kaggle/working/training_history.csv")
print("/kaggle/working/validation_predictions.csv")

Saved files:
/kaggle/working/best_efficientnet_b0_wang1000.pth
/kaggle/working/training_history.csv
/kaggle/working/validation_predictions.csv
